In [0]:
import pandas as pd

In [0]:
df_ratings_pd =pd.read_csv("/Volumes/movie_catalog/movie_schema/movie_vol/User_ratings.gzip", compression="gzip", parse_dates=["Review_date"])

In [0]:
df_ratings = spark.createDataFrame(df_ratings_pd)
display(df_ratings.show(5))

In [0]:
df_ratings.printSchema()

In [0]:
from pyspark.sql.functions import col, lit,year,month,lpad,concat,StringType
df_ratings = df_ratings.withColumn("Rev_year",year(col("Review_date"))).withColumn("Rev_months",lpad(month(col("Review_date")),2,"0"))
df_ratings = df_ratings.withColumn("Review_ym",concat(col("Rev_year").cast(StringType()),col("Rev_months").cast(StringType())) )
display(df_ratings.sort("Review_ym").show(5))

In [0]:
df_ratings.select("Review_ym").distinct().count()

In [0]:
from pyspark.sql.functions import count,avg
df_ratings.groupby("Rev_year","Rev_months").agg(count("Rating").alias("Rating_count"),avg("Rating").alias("Avg_Rating")).orderBy("Rev_year","Rev_months").show()

In [0]:
%python
catalog = "movie_catalog"
schema = dbName = db = "movie_schema"
volume_name = "monthwise_rating_data"

spark.sql(f'CREATE CATALOG IF NOT EXISTS `{catalog}`')
spark.sql(f'USE CATALOG `{catalog}`')
spark.sql(f'CREATE SCHEMA IF NOT EXISTS `{catalog}`.`{schema}`')
spark.sql(f'USE SCHEMA `{schema}`')
spark.sql(f'CREATE VOLUME IF NOT EXISTS `{catalog}`.`{schema}`.`{volume_name}`')
volume_folder =  f"/Volumes/{catalog}/{db}/{volume_name}"

In [0]:
df_ratings.select("Rev_year").distinct().collect()

In [0]:
df_to_write =df_ratings.select("MovieID","Rating","Review_ym").filter("Rev_year < 2003")
df_to_write_latter =df_ratings.select("MovieID","Rating","Review_ym").filter("Rev_year >= 2003")
print(df_to_write.count())
df_to_write_latter.count()

In [0]:
df_ratings.groupby("Rev_year").agg(count("Rating").alias("Count of Ratings")).show()

In [0]:
#for i in df_to_write.collect():
#   print(row['Review_ym'])



#df_to_write.repartition(30).write.format("csv").mode("overwrite").option("header", "True").save(volume_folder)

In [0]:
df_ratings.show()

In [0]:
lst =list(df_to_write.select("Review_ym").distinct())
print(lst)


In [0]:
df_to_write.select("Review_ym").distinct()

In [0]:
from pyspark.sql.functions import col
import os
import shutil
def lpad(value, length, pad_char):
    return str(value).rjust(length, str(pad_char))
def replace(value,sstring,tstring):
    return value.replace(sstring,tstring)
cnt =1 
for row in df_to_write.select("Review_ym").distinct().orderBy("Review_ym").collect():
    print(row['Review_ym'],df_to_write.filter(col("Review_ym") == row['Review_ym']).count(),cnt)
    df_to_write.filter(col("Review_ym") == row['Review_ym']).coalesce(1).write.format("csv").mode("overwrite").option("header", "True").save(volume_folder)
    #if cnt <10:
        #cnt = cnt +1
        #continue;
    for filename in os.listdir(volume_folder):
        if filename.endswith(".csv"):
            newfilename =replace(filename,"-00000-","-"+lpad(str(cnt),5,0)+"-")
            shutil.move(os.path.join(volume_folder,filename),os.path.join('/Volumes/movie_catalog/movie_schema/movie_user_ratings',newfilename))
    if cnt > 11:
        break;
    cnt = cnt +1    
    #shutil.rmtree(volume_folder) 


from pyspark.sql.functions import replace
filename ="""/Volumes/movie_catalog/movie_schema/movie_user_ratings/part-00000-tid-6008058980271264566-2173be1d-195c-419b-b451-83830226ec84-327-1-c000.csv"""
cnt = 1

def lpad(value, length, pad_char):
    return str(value).rjust(length, str(pad_char))
def replace(value,sstring,tstring):
    return value.replace(sstring,tstring)
print(lpad(str(cnt),5,0))
print(replace(filename,"-00000-","-"+lpad(str(cnt),5,0)+"-"))